# 🤖 Tahap 4, 5 & 6: Data Transformation, Data Mining & Visualisasi
Notebook ini berisi proses inti:
- **Tahap 4**: Transformasi data ke Matriks Biner (One-Hot Encoding).
- **Tahap 5**: Eksekusi Algoritma Apriori & Pembangkitan Association Rules.
- **Tahap 6**: Visualisasi hasil (Heatmap, Bar Chart, Network Graph).
- **Rangkuman & Kesimpulan**.

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Load data preprocessing
df_pre = pd.read_csv('../data/mba-laundry-transaction-preprocessing-20260704.csv')

# Parse Variasi menjadi list item
transactions = df_pre['Variasi'].apply(lambda x: [item.strip() for item in str(x).split(',')]).tolist()
print(f'Total transaksi: {len(transactions)}')
print(f'Contoh 5 keranjang pertama: {transactions[:5]}')

---
## Tahap 4: Data Transformation (One-Hot Encoding)

### 4.1. Membentuk Matriks Boolean & Biner (Sebagian Data)
Menampilkan cuplikan hasil transformasi dalam format True/False dan versi biner (1/0).

In [ ]:
# Transformasi dengan TransactionEncoder
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print('=== Matriks Boolean (True/False) — 10 Baris Pertama ===')
display(df_encoded.head(10))

print('\n=== Matriks Biner (1/0) — 10 Baris Pertama ===')
display(df_encoded.astype(int).head(10))

💡 **Pembahasan:**
- TransactionEncoder mengubah setiap keranjang belanja menjadi vektor Boolean.
- Nilai `True` (1) berarti layanan tersebut ada di dalam keranjang, `False` (0) berarti tidak.
- Matriks biner inilah yang menjadi bahan bakar murni algoritma Apriori.

### 4.2. Menampilkan Seluruh Matriks Biner (Tampilan Penuh)
Bukti konkret bahwa seluruh 273 transaksi berhasil ditransformasi.

In [ ]:
# Tampilkan seluruh matriks biner
pd.set_option('display.max_rows', None)
display(df_encoded.astype(int))
pd.reset_option('display.max_rows')
print(f'\nTotal baris: {df_encoded.shape[0]}, Total kolom (layanan): {df_encoded.shape[1]}')

💡 **Pembahasan:**
- Tabel di atas menampilkan seluruh 273 baris matriks biner tanpa dipotong.
- Ini membuktikan bahwa proses One-Hot Encoding berjalan sempurna untuk seluruh transaksi.

---
## Tahap 5: Data Mining (Algoritma Apriori & Association Rules)

**Rumus yang digunakan:**
- `Support(A → B) = Transaksi mengandung A dan B / Total Transaksi`
- `Confidence(A → B) = Transaksi mengandung A dan B / Transaksi mengandung A`
- `Lift(A → B) = Confidence(A → B) / Support(B)`

**Parameter:**
- Minimum Support: **4%** (0.04)
- Minimum Confidence: **30%** (0.30)

In [ ]:
# Jalankan Algoritma Apriori
frequent_itemsets = apriori(df_encoded, min_support=0.04, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

print(f'Frequent itemsets ditemukan: {len(frequent_itemsets)}')
print('\n=== Frequent Itemsets ===')
display(frequent_itemsets.sort_values('support', ascending=False))

💡 **Pembahasan:**
- Algoritma Apriori berhasil menemukan sejumlah *frequent itemsets* yang melewati ambang batas support 4%.
- Kolom `length` menunjukkan jumlah item dalam setiap itemset (1-item, 2-item, dst).

In [ ]:
# Bangkitkan Association Rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.3)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Association rules ditemukan: {len(rules)}')
print('\n=== Top Association Rules (Diurutkan berdasarkan Lift) ===')
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

💡 **Pembahasan:**
- Tabel aturan asosiasi di atas sudah diurutkan berdasarkan **Lift Ratio** tertinggi.
- Aturan dengan Lift > 1 menandakan asosiasi yang valid (bukan kebetulan acak).
- Kombinasi perlengkapan tidur (Sprei & Bed Cover) memiliki pola keterikatan tertinggi.

---
## Tahap 6: Knowledge Representation (Visualisasi Pola)

### 6.1. Heatmap Korelasi Layanan

In [ ]:
# Heatmap Korelasi
heatmap_data = rules.copy()
heatmap_data['antecedent'] = heatmap_data['antecedents'].apply(lambda x: ', '.join(list(x)))
heatmap_data['consequent'] = heatmap_data['consequents'].apply(lambda x: ', '.join(list(x)))

mask = (heatmap_data['antecedents'].apply(len) == 1) & (heatmap_data['consequents'].apply(len) == 1)
heatmap_filtered = heatmap_data[mask]

pivot = heatmap_filtered.pivot_table(index='antecedent', columns='consequent', values='lift', aggfunc='mean')

plt.figure(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu', linewidths=0.5, square=True)
plt.title('Heatmap Korelasi Layanan (Lift Ratio)', fontsize=14, fontweight='bold')
plt.xlabel('Consequent (Layanan Tujuan)', fontsize=11)
plt.ylabel('Antecedent (Layanan Asal)', fontsize=11)
plt.tight_layout()
plt.show()

💡 **Pembahasan:**
- Warna biru gelap menunjukkan pasangan sejati (asosiasi kuat, Lift tinggi).
- Warna kuning pucat / kosong menunjukkan pasangan musuh alami (substitusi).
- Cuci Kering dan Setrika memiliki keterikatan sangat kuat.

### 6.2. Bar Chart Top Association Rules

In [ ]:
# Bar Chart Top Rules
top_rules = rules.head(10).copy()
top_rules['rule_label'] = top_rules.apply(
    lambda row: f"{', '.join(list(row['antecedents']))} → {', '.join(list(row['consequents']))}",
    axis=1
)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=top_rules, y='rule_label', x='lift', palette='magma', orient='h')

for i, (_, row) in enumerate(top_rules.iterrows()):
    ax.text(row['lift'] + 0.02, i, f"Lift: {row['lift']:.2f}", va='center', fontsize=9, fontweight='bold')

plt.title('Top 10 Association Rules (Berdasarkan Lift Ratio)', fontsize=14, fontweight='bold')
plt.xlabel('Lift Ratio', fontsize=11)
plt.ylabel('Aturan Asosiasi', fontsize=11)
plt.tight_layout()
plt.show()

💡 **Pembahasan:**
- Semakin panjang batangnya, semakin kuat aturannya.
- Baris terpanjang di atas adalah *Golden Rule* — kombinasi paten paling direkomendasikan untuk paket promosi silang (*Cross-Selling*).

### 6.3. Network Graph (Peta Arus Kasir)

In [ ]:
# Network Graph
top_net = rules.head(10)
G = nx.DiGraph()

for _, row in top_net.iterrows():
    ante = ', '.join(list(row['antecedents']))
    cons = ', '.join(list(row['consequents']))
    G.add_edge(ante, cons, weight=row['lift'])

plt.figure(figsize=(12, 9))
pos = nx.spring_layout(G, k=2.5, seed=42)
edges = G.edges(data=True)
weights = [d['weight'] for _, _, d in edges]
max_w, min_w = max(weights), min(weights)
edge_widths = [1 + 4 * (w - min_w) / (max_w - min_w + 0.01) for w in weights]

nx.draw_networkx_nodes(G, pos, node_color='#00796B', node_size=2500, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', font_color='white')
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color='#E64A19',
                       arrows=True, arrowsize=20, connectionstyle='arc3,rad=0.15', alpha=0.8)
edge_labels = {(u, v): f"Lift: {d['weight']:.2f}" for u, v, d in edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)

plt.title('Network Graph: Peta Arus Rekomendasi Kasir', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

💡 **Pembahasan:**
- Node (lingkaran) adalah layanan, panah adalah pola asosiasi.
- Jika panah dari Sprei menuju Bed Cover, artinya kasir harus selalu menawari cuci Bed Cover ketika pelanggan membawa Sprei.
- Grafik ini dapat digunakan sebagai SOP operasional kasir.

---
## 📌 Rangkuman & Kesimpulan

**Temuan Utama:**
1. **Cash Cow**: Layanan *Cuci Kering & Setrika* adalah urat nadi perusahaan dengan omset tertinggi.
2. **Golden Rule**: Kombinasi perlengkapan tidur (Sprei & Bed Cover) memiliki Lift Ratio tertinggi.
3. **Substitusi**: Layanan tertentu bersifat menggantikan (Lift < 1), sehingga pantang di-bundling.

**Rekomendasi Strategi Bisnis:**
1. Terapkan promo silang **'Bed Room Bundle'** (Sprei + Bed Cover) karena probabilitas pelanggan membeli keduanya sangat tinggi.
2. Jangan mem-bundling layanan yang bersifat substitusi.
3. Pertahankan layanan *Cuci Kering & Setrika* sebagai produk unggulan utama.